# **OpenAI Agents SDK**

Building an agent system for generating cold sales outreach emails:
- Agent workflow
- Use of tools to call functions
- Agent collaboration via Tools and Handoffs

https://openai.github.io/openai-agents-python/


## Setup

In [25]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio

from openai import AsyncOpenAI
from pydantic import BaseModel

load_dotenv()

True

### We can send emails using SendGrid's API. 

Create the API Key in https://sendgrid.com/ and enable an email in Sender Authentication.

In [26]:
# Check emails are working
""" 
def send_test_email():
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("plvital422@gmail.com")
    to_email = To("pedro.vital@ufms.br")
    content = Content("text/plain", "This is an important test email")
    mail = Mail(from_email, to_email, "Test email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print(response.status_code)

send_test_email()
 """

' \ndef send_test_email():\n    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get(\'SENDGRID_API_KEY\'))\n    from_email = Email("plvital422@gmail.com")\n    to_email = To("pedro.vital@ufms.br")\n    content = Content("text/plain", "This is an important test email")\n    mail = Mail(from_email, to_email, "Test email", content).get()\n    response = sg.client.mail.send.post(request_body=mail)\n    print(response.status_code)\n\nsend_test_email()\n '

### It's easy to use any models with OpenAI compatible endpoints

In [27]:
groq_api_key = os.getenv('GROQ_API_KEY')
openai_api_key = os.getenv('OPENAI_API_KEY')
if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

Groq API Key exists and begins gsk_
OpenAI API Key exists and begins sk-proj-


In [ ]:
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

llama3_3 = OpenAIChatCompletionsModel(model="llama-3.3-70b-versatile", openai_client=groq_client)
qwen = OpenAIChatCompletionsModel(model="qwen/qwen3-32b", openai_client=groq_client)


Results might be compromised due to the model. Better to use gpt-4o-mini.

### Enable Tracing

In [29]:
os.environ["OPENAI_TRACING"] = "true"

## 1. Agent workflow

#### System Prompt

In [30]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

#### Agents

In [31]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model=llama3_3
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model=llama3_3
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model=llama3_3
)

To use an OpenAI model, you just need to pass its name as string (eg. "gpt-4o-mini")

In [32]:
sales_agent1

Agent(name='Professional Sales Agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You are a sales agent working for ComplAI, a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. You write professional, serious cold emails.', prompt=None, handoffs=[], model=<agents.models.openai_chatcompletions.OpenAIChatCompletionsModel object at 0x76862a4b9590>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None, retry=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

#### Test Run 

In [33]:
# Run with Runner.run(agent, prompt) then print final_output
with trace("First sales agent"):
    result = await Runner.run(sales_agent1, "Write a cold sales email")
    print(result.final_output)

Subject: Simplify SOC 2 Compliance with ComplAI

Dear [Recipient's Name],

I came across your company, [Company Name], and was impressed by the innovative work you're doing in [Industry/Field]. As a leader in the industry, I'm sure you understand the importance of maintaining the trust of your customers and stakeholders by ensuring the security and integrity of your systems and data.

SOC 2 compliance is a critical component of this effort, but the process of achieving and maintaining it can be complex, time-consuming, and resource-intensive. That's where ComplAI comes in. Our AI-powered SaaS tool is designed to simplify SOC 2 compliance and audit preparation, helping you reduce risk, increase efficiency, and demonstrate your commitment to security and compliance.

With ComplAI, you'll be able to:

* Automate compliance monitoring and reporting
* Identify and remediate gaps in your controls
* Streamline audit preparation and reduce the time and cost associated with it
* Demonstrate com

In [34]:
Runner.run(sales_agent1, "Write a cold sales email")
# It's a coroutine, so we need to run it with asyncio.run(), but we are in a Jupyter notebook, so just using await will work

<coroutine object Runner.run at 0x768647f5f040>

Trace in OpenAI Plataform: https://platform.openai.com/logs?api=traces

#### Test Run in stream mode

In [35]:
result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)


Subject: Streamline SOC 2 Compliance with ComplAI

Dear [Recipient's Name],

I hope this email finds you well. As a [Recipient's Job Title] at [Company Name], I'm sure you understand the importance of maintaining the highest standards of security and compliance in today's rapidly evolving tech landscape.

As a trusted provider of [Company Name]'s services, ensuring the confidentiality, integrity, and availability of sensitive data is paramount. SOC 2 compliance is a critical component of this effort, demonstrating to your customers, partners, and stakeholders that your organization prioritizes data security and adheres to stringent industry standards.

ComplAI is a cutting-edge SaaS solution designed to simplify and accelerate the SOC 2 compliance process. Our AI-powered platform helps organizations like yours navigate the complexities of audit preparation, providing a comprehensive framework for identifying and mitigating risks, automating evidence collection, and ensuring continuous 

#### Multiple agents Run

In [36]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    
outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

Subject: Streamline Your SOC 2 Compliance with ComplAI

Dear [Recipient's Name],

I hope this email finds you well. As a [Recipient's Job Title] at [Company Name], I'm sure you're no stranger to the importance of maintaining SOC 2 compliance. The complexity and ever-evolving nature of regulatory requirements can be daunting, taking away from the time and resources you could be dedicating to driving business growth.

At ComplAI, we understand the challenges that come with ensuring SOC 2 compliance. That's why we've developed a cutting-edge SaaS tool powered by artificial intelligence, designed to simplify and accelerate your compliance journey. Our platform helps organizations like yours navigate the intricacies of SOC 2 requirements, ensuring that you're always audit-ready.

With ComplAI, you can:

* Automate compliance workflows and reduce manual effort
* Identify and remediate gaps in your current controls
* Receive real-time guidance and recommendations from our AI-powered engine
* 

#### Agent as **evaluator** of best email

In [37]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model=llama3_3
)

In [38]:
message = "Write a cold sales email"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")


Best sales email:
Subject: SOC2 Audit: From Nightmare to Breeze

Dear [Recipient's Name],

I hope this email finds you well and that your SOC2 audit isn't keeping you up at night (but let's be real, it probably is).

As someone who's likely no stranger to the thrill of compliance, I'm here to introduce you to ComplAI - the ultimate SOC2 sidekick. Our AI-powered SaaS tool is like having a team of compliance superheroes at your fingertips, minus the fancy suits and awkward team-building exercises.

With ComplAI, you'll be able to:

Automate tedious compliance tasks (because who doesn't love a good robot?)
Get real-time monitoring and alerts for any potential issues (think of it as having a personal compliance watchdog)
Spend more time on high-level strategy and less on audit prep (you know, the fun stuff)

Our tool has already helped numerous companies like yours streamline their SOC2 compliance and ace their audits. And the best part? It's ridiculously easy to use, even for those who ar

## 2. Use of Tools

#### Define Tool

In [39]:
@function_tool
def send_email(body: str):
    """ Send out an email with the given body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("plvital422@gmail.com")
    to_email = To("pedro.vital@ufms.br")
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Sales email", content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

send_email

FunctionTool(name='send_email', description='Send out an email with the given body to all sales prospects', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7686441a6510>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

#### Agent as Tool

In [40]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

tools


[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x76862a354290>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._Fa

#### Sales Manager Agent

In [42]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""


sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=qwen)

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)

## 3. Handoffs

Handoffs and Agents-as-tools are similar: In both cases, an Agent can collaborate with another Agent
- With tools, control passes back
- With handoffs, control passes across

In [43]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model=llama3_3)
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model=llama3_3)
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")


In [44]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("plvital422@gmail.com")
    to_email = To("pedro.vital@ufms.br")
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

tools = [subject_tool, html_tool, send_html_email]
tools

[FunctionTool(name='subject_writer', description='Write a subject for a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7686462b6de0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False),
 FunctionTool(name='html_converter', description='Convert a text email body to an HTML email body', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties'

In [49]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="email_manager",
    instructions=instructions,
    tools=tools,
    model=llama3_3,
    handoff_description="Convert an email to HTML and send it")

handoffs = [emailer_agent]
handoffs

[Agent(name='email_manager', handoff_description='Convert an email to HTML and send it', tools=[FunctionTool(name='subject_writer', description='Write a subject for a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7686462b6de0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False), FunctionTool(name='html_converter', description='Convert a text email body to an HTML email body', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}

In [ ]:
sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini")

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

## 4. Guardrails

In [66]:
class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str

guardrail_agent = Agent( 
    name="Name check",
    instructions="Check if the user is including someone's personal name in what they want you to do.",
    output_type=NameCheckOutput,
    model=qwen
)


In [67]:
@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    # the agent is going to return a NameCheckOutput object
    is_name_in_message = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(output_info={"found_name": result.final_output},tripwire_triggered=is_name_in_message)


There is input guardrail and output guardrail. To use the output guardrail we're going to pass the output rather than the message. Otherwise, it's exactly the same.

In [ ]:
careful_sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=[emailer_agent],
    model="gpt-4o-mini",
    input_guardrails=[guardrail_against_name]
    )

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, message)


In [ ]:
message = "Send out a cold sales email addressed to Dear CEO from Head of Business Development"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, message)